# 第 7 章互动实验：立方体的针孔投影

本实验用合成三维点和 OpenCV `projectPoints` 作为可视化基线，不给出 `7-1_针孔投影.py` 的矩阵实现。调节深度和旋转角，观察近大远小及有效深度。

In [ ]:
%matplotlib inline
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

K = np.array([[520.0, 0, 320.0], [0, 520.0, 240.0], [0, 0, 1.0]])
dist = np.zeros(5)
cube = np.float32([
    [-1,-1,-1], [1,-1,-1], [1,1,-1], [-1,1,-1],
    [-1,-1, 1], [1,-1, 1], [1,1, 1], [-1,1, 1],
])
edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

In [ ]:
def show_projection(depth=7.0, yaw_deg=25):
    yaw = np.deg2rad(yaw_deg)
    R = np.array([[np.cos(yaw), 0, np.sin(yaw)], [0, 1, 0], [-np.sin(yaw), 0, np.cos(yaw)]], dtype=np.float64)
    rvec, _ = cv2.Rodrigues(R)
    tvec = np.array([0.0, 0.0, depth])
    camera_points = (R @ cube.astype(np.float64).T).T + tvec
    pixels, _ = cv2.projectPoints(cube, rvec, tvec, K, dist)
    pixels = pixels.reshape(-1, 2)
    valid = camera_points[:, 2] > 0
    fig, ax = plt.subplots(figsize=(7, 5.3))
    ax.set_xlim(0, 640); ax.set_ylim(480, 0); ax.set_aspect('equal')
    ax.set_facecolor('#101820')
    for a, b in edges:
        if valid[a] and valid[b]: ax.plot(pixels[[a,b],0], pixels[[a,b],1], color='#57d6ff', linewidth=2)
    ax.scatter(pixels[valid,0], pixels[valid,1], c='#ffcc66', s=35)
    span = np.ptp(pixels[valid], axis=0) if np.any(valid) else np.array([np.nan, np.nan])
    ax.set_title(f'depth={depth:.1f}, yaw={yaw_deg}°，投影跨度 {span[0]:.1f}×{span[1]:.1f} px，有效点 {valid.sum()}/8')
    ax.set_xlabel('u'); ax.set_ylabel('v'); ax.grid(alpha=0.2); plt.show()

interact(
    show_projection,
    depth=FloatSlider(min=1.5, max=14.0, step=0.5, value=7.0),
    yaw_deg=IntSlider(min=-75, max=75, step=5, value=25),
);

## 破坏实验与记录

- 把深度从 14 降到 2，记录包围盒尺度变化和无效深度。
- 修改 `K` 中焦距或主点，区分缩放与整体平移。
- 思考同一条相机射线上的两个点为何投到同一像素。

把 `K`、位姿、深度范围、投影跨度与失败原因写入 `实验记录.md`。